In [1]:
# Aggregate capacity and capacity addition from the ESM by location (works for all SCMs)

from zen_garden.postprocess.results import Results
import pandas as pd
from pathlib import Path

# Configuration: adjust name, technologies, and locations according to your dataset
#specific locations are treated separately and rest is summed as ROE
dataset_name = "energy_transition_example"
selected_technologies = {"heat_pump", "photovoltaics", "wind_onshore"}
specific_locations = {"DE", "CH", "DK", "SE", "UK", "NL"}

base_path = Path("outputs")
results_path = base_path / dataset_name
output_dir = Path("csv_output") / dataset_name


output_dir.mkdir(parents=True, exist_ok=True)

# Load Results
res = Results(results_path)
capacity = res.get_full_ts("capacity").reset_index()
capacity_addition = res.get_full_ts("capacity_addition").reset_index()

# Filter by selected technologies
capacity = capacity[capacity["technology"].isin(selected_technologies)]
capacity_addition = capacity_addition[capacity_addition["technology"].isin(selected_technologies)]

# Save filtered raw data
capacity.round(4).to_csv(output_dir / "capacity.csv", index=False)
capacity_addition.round(4).to_csv(output_dir / "capacity_addition.csv", index=False)

# Identify rest-of-Europe (ROE) locations
all_locations = set(capacity["location"].unique())
roe_locations = all_locations - specific_locations

# Helper function for aggregation
def aggregate_by_location(df, location_set, location_name):
    return (
        df[df["location"].isin(location_set)]
        .groupby(["technology", "capacity_type"])
        .sum(numeric_only=True)
        .assign(location=location_name)
    )

# Perform aggregation
def perform_aggregation(df):
    parts = [aggregate_by_location(df, {loc}, loc) for loc in specific_locations]
    parts.append(aggregate_by_location(df, roe_locations, "ROE"))
    return pd.concat(parts).reset_index()

capacity_aggregated = perform_aggregation(capacity)
capacity_addition_aggregated = perform_aggregation(capacity_addition)

# Save aggregated data
capacity_aggregated.round(4).to_csv(output_dir / "capacity_aggregated_by_location.csv", index=False)
capacity_addition_aggregated.round(4).to_csv(output_dir / "capacity_addition_aggregated_by_location.csv", index=False)

print(f"CSV files for dataset '{dataset_name}' saved in: {output_dir}")


CSV files for dataset 'energy_transition_example' saved in: csv_output\energy_transition_example


In [2]:
# rename_year_columns.py
"""
Renames numeric year columns (e.g. "0", "1", ...) in aggregated CSVs
to actual years (e.g. "2025", "2030", ...) using system.json config.
Also reorders columns so 'location' is the third column.
"""

import json
import pandas as pd
from pathlib import Path

# CONFIGURATION
dataset_name = "energy_transition_example"
results_path = Path("outputs") / dataset_name
output_dir = Path("csv_output") / dataset_name

# Load system.json and construct rename map
system_path = results_path / "system.json"
with open(system_path, "r") as f:
    config = json.load(f)

ref_year = config["reference_year"]
interval = config["interval_between_years"]
n_years = config["optimized_years"]

year_map = {str(i): str(ref_year + i * interval) for i in range(n_years)}

def rename_year_columns(csv_file):
    df = pd.read_csv(csv_file)
    df.rename(columns={col: year_map[col] for col in df.columns if col in year_map}, inplace=True)

    # Reorder columns to put location third
    base_cols = ["technology", "capacity_type", "location"]
    rest = [c for c in df.columns if c not in base_cols]
    df = df[base_cols + rest]

    df.to_csv(csv_file, index=False)
    print(f"Updated: {csv_file.name}")

# Apply to both aggregated files
rename_year_columns(output_dir / "capacity_aggregated_by_location.csv")
rename_year_columns(output_dir / "capacity_addition_aggregated_by_location.csv")

Updated: capacity_aggregated_by_location.csv
Updated: capacity_addition_aggregated_by_location.csv


In [10]:
#specific for wind (write values to demand_yearly_variation.csv)
#import pandas as pd
import shutil
#from pathlib import Path
#import json

# CONFIGURATION
#dataset_name = "energy_transition_ref_2021"

# Base paths
#base_path = Path(r".\outputs")
#output_dir = Path("./CSV_output") / dataset_name
esm_capacity_path = output_dir / "capacity_addition_aggregated_by_location.csv"

original_dir = Path(r"C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_Wind\Data_WT")
new_dir = Path(r"C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_Wind\Data_WT_new")
original_dyv_path = original_dir / "set_carriers" / "Turbine" / "demand_yearly_variation.csv"
target_dyv_path = new_dir / "set_carriers" / "Turbine" / "demand_yearly_variation.csv"

# --- Step 1: Clone the original directory ---
if new_dir.exists():
    print(f"Directory already exists: {new_dir}")
else:
    shutil.copytree(original_dir, new_dir)
    print(f"Created clone of wind model input: {new_dir}")

# --- Step 2: Load year labels from system.json ---
system_path = base_path.parent / dataset_name / "system.json"
with open(system_path, "r") as f:
    system_config = json.load(f)

reference_year = system_config["reference_year"]
interval = system_config["interval_between_years"]
optimized_years = system_config["optimized_years"]
year_labels = [str(reference_year + i * interval) for i in range(optimized_years)]

# --- Step 3: Load original demand_yearly_variation to preserve all nodes ---
df_existing = pd.read_csv(original_dyv_path)

# --- Step 4: Prepare updated wind data from ESM ---
country_map = {
    "DE": "DEU",
    "DK": "DNK",
    "CH": "CHE",
    "SE": "SWE",
    "NL": "NLD",
    "UK": "GBR",
    "ROE": "ROE",
    "ROW": "ROW",
}

df = pd.read_csv(esm_capacity_path)
df_wind = df[df["technology"].isin(["wind_onshore"])]
df_grouped = df_wind.groupby(["location"]).sum(numeric_only=True).reset_index()
df_grouped["node"] = df_grouped["location"].map(country_map)
df_grouped = df_grouped[df_grouped["node"].notna()].drop(columns=["location"])

# Transpose to match demand_yearly_variation structure
df_new = df_grouped.set_index("node").T
df_new.index.name = "year"
df_new = df_new.reset_index()

# Use existing index (years) instead of slicing year_labels
df_new["year"] = df_new["year"].astype(str)
df_existing["year"] = df_existing["year"].astype(str)


# Set year as index for merging
df_new = df_new.set_index("year")
df_existing = df_existing.set_index("year")

# --- Step 5: Merge values only for matching columns ---
updated_cols = [col for col in df_new.columns if col in df_existing.columns]

for col in updated_cols:
    df_existing[col] = df_new[col].combine_first(df_existing[col])

# --- Step 6: Save final file ---
df_result = df_existing.reset_index()
df_result.to_csv(target_dyv_path, index=False)
print(f"Final demand_yearly_variation.csv written to:\n{target_dyv_path}")


Directory already exists: C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_Wind\Data_WT_new
Final demand_yearly_variation.csv written to:
C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_Wind\Data_WT_new\set_carriers\Turbine\demand_yearly_variation.csv
